# Reproduce Table 2 with pandas and statsmodels
This notebook loads `data/cleaned_data.csv`, reconstructs the same regression specifications used in `3-Analysis.do`, and presents the resulting table in pandas.


In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
pd.options.display.float_format = '{:.3f}'.format
data_path = 'data/cleaned_data.csv'
df = pd.read_csv(data_path, dtype={'code': str})
print(f'Loaded {len(df):,} rows from {data_path}')
df.head()


Loaded 13,818 rows from data/cleaned_data.csv


,id_in_session,code,index_in_pages,current_page_name,time_started,payoff,consent,treatment,age_temp,gender,...,mean_prob_rep_temp,mean_prob_rep,mean_prob_dem_temp,mean_prob_dem,mean_prob_net,mean_prob_net_mean,mean_prob_net_sd,mean_prob_net_z,out_of_range_q,out_of_range
0,213,00d576h7,155,Results,2018-06-25 20:15:41.255206+00:00,0,NaN,SND,NaN,NaN,...,NaN,0.517,NaN,0.700,-0.183,-0.064,0.184,-0.645,0,0
1,213,00d576h7,155,Results,2018-06-25 20:15:41.255206+00:00,0,NaN,SND,NaN,NaN,...,NaN,0.517,NaN,0.700,-0.183,-0.064,0.184,-0.645,0,0
2,213,00d576h7,155,Results,2018-06-25 20:15:41.255206+00:00,0,NaN,SND,NaN,NaN,...,NaN,0.517,0.700,0.700,-0.183,-0.064,0.184,-0.645,0,0
3,213,00d576h7,155,Results,2018-06-25 20:15:41.255206+00:00,0,NaN,SND,NaN,NaN,...,0.517,0.517,NaN,0.700,-0.183,-0.064,0.184,-0.645,0,0
4,213,00d576h7,155,Results,2018-06-25 20:15:41.255206+00:00,0,NaN,SND,NaN,NaN,...,NaN,0.517,NaN,0.700,-0.183,-0.064,0.184,-0.645,0,0


In [2]:
# Prepare analysis variables and data types
bool_cols = [
    'pro_party', 'anti_party', 'true_news', 'fake_news', 'neutral_news',
    'message_greater', 'message_less', 'politicized_news', 'red_state',
    'religious_group', 'white', 'black', 'asian', 'latino', 'male'
]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

df['anti_party_drop'] = df['anti_party']
df['pro_party_str'] = df['pro_party'] * df['abs_net_party']
df['round_number'] = df['round_number'].astype('category')
df['topic_id'] = df['topic_id'].astype('category')
df['code'] = df['code'].astype('category')

# Keep the same sample logic as the Stata specs
df['only_political_news'] = (df['pro_party'] + df['anti_party']) == 1

df[['age', 'edu', 'log_inc']] = df[['age', 'edu', 'log_inc']].astype(float)

df[['pro_party', 'anti_party', 'true_news', 'neutral_news', 'message_greater', 'message_less']].head()


C:\Users\Hex\AppData\Local\Temp\ipykernel_12484\490892715.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['anti_party_drop'] = df['anti_party']
C:\Users\Hex\AppData\Local\Temp\ipykernel_12484\490892715.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['only_political_news'] = (df['pro_party'] + df['anti_party']) == 1


,pro_party,anti_party,true_news,neutral_news,message_greater,message_less
0,0,0,0,1,0,1
1,0,0,1,0,0,0
2,0,1,0,0,0,1
3,1,0,1,0,1,0
4,0,0,1,0,1,0


In [3]:
def run_regression(formula, data, cluster_col='code'): 
    model = smf.ols(formula=formula, data=data).fit(
        cov_type='cluster', cov_kwds={'groups': data[cluster_col]}
    )
    return model

specs = [
    {
        'label': '(1)',
        'formula': 'prob_true ~ pro_party + net_party + C(round_number) + C(topic_id) + white + black + asian + latino + age + male + edu + red_state + religious_group + log_inc',
        'mask': df['only_political_news'],
        'question_fe': True,
        'round_fe': True,
        'subject_fe': False,
        'subject_controls': True,
        'neutral_news': False,
    },
    {
        'label': '(2)',
        'formula': 'prob_true ~ pro_party + C(round_number) + C(topic_id) + C(code)',
        'mask': df['only_political_news'],
        'question_fe': True,
        'round_fe': True,
        'subject_fe': True,
        'subject_controls': False,
        'neutral_news': False,
    },
    {
        'label': '(3)',
        'formula': 'prob_true ~ pro_party + pro_party_str + abs_net_party:C(round_number) + abs_net_party:C(topic_id) + C(round_number) + C(topic_id) + C(code)',
        'mask': df['only_political_news'],
        'question_fe': True,
        'round_fe': True,
        'subject_fe': True,
        'subject_controls': False,
        'neutral_news': False,
    },
    {
        'label': '(4)',
        'formula': 'prob_true ~ pro_party + anti_party + C(round_number) + C(code)',
        'mask': (df['net_party'] != 0) & ((df['politicized_news'] + df['neutral_news']) == 1) & ((df['message_greater'] + df['message_less']) == 1),
        'question_fe': False,
        'round_fe': True,
        'subject_fe': True,
        'subject_controls': False,
        'neutral_news': True,
    },
    {
        'label': '(5)',
        'formula': 'prob_true ~ true_news + C(round_number) + C(topic_id) + C(code)',
        'mask': df['only_political_news'],
        'question_fe': True,
        'round_fe': True,
        'subject_fe': True,
        'subject_controls': False,
        'neutral_news': False,
    },
    {
        'label': '(6)',
        'formula': 'prob_true ~ pro_party + true_news + C(round_number) + C(topic_id) + C(code)',
        'mask': df['only_political_news'],
        'question_fe': True,
        'round_fe': True,
        'subject_fe': True,
        'subject_controls': False,
        'neutral_news': False,
    },
]
for spec in specs:
    data = df.loc[spec['mask'], :].copy()
    spec['model'] = run_regression(spec['formula'], data)
    spec['n'] = len(data)
    spec['mean'] = data['prob_true'].mean()

specs


[{'label': '(1)',
  'formula': 'prob_true ~ pro_party + net_party + C(round_number) + C(topic_id) + white + black + asian + latino + age + male + edu + red_state + religious_group + log_inc',
  'mask': 0        False
  1        False
  2         True
  3         True
  4        False
           ...  
  13813     True
  13814     True
  13815    False
  13816     True
  13817    False
  Name: only_political_news, Length: 13818, dtype: bool,
  'question_fe': True,
  'round_fe': True,
  'subject_fe': False,
  'subject_controls': True,
  'neutral_news': False,
  'model': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x2c059826900>,
  'n': 7902,
  'mean': np.float64(0.5741964080422678)},
 {'label': '(2)',
  'formula': 'prob_true ~ pro_party + C(round_number) + C(topic_id) + C(code)',
  'mask': 0        False
  1        False
  2         True
  3         True
  4        False
           ...  
  13813     True
  13814     True
  13815    False
  13816     True
  13817    Fa

In [4]:
rows = [
    'Pro-Party News',
    'Pro-Party News SE',
    'Partisanship x Pro-Party',
    'Partisanship x Pro-Party SE',
    'Anti-Party News',
    'Anti-Party News SE',
    'True News',
    'True News SE',
    'Question FE',
    'Round FE',
    'Subject FE',
    'Subject controls',
    'Neutral News',
    'Observations',
    'R2',
    'Mean',
]
table_data = {spec['label']: [] for spec in specs}
for spec in specs:
    model = spec['model']
    table_data[spec['label']].extend([
        model.params.get('pro_party', np.nan),
        model.bse.get('pro_party', np.nan),
        model.params.get('pro_party_str', np.nan),
        model.bse.get('pro_party_str', np.nan),
        model.params.get('anti_party', np.nan),
        model.bse.get('anti_party', np.nan),
        model.params.get('true_news', np.nan),
        model.bse.get('true_news', np.nan),
        'Yes' if spec['question_fe'] else 'No',
        'Yes' if spec['round_fe'] else 'No',
        'Yes' if spec['subject_fe'] else 'No',
        'Yes' if spec['subject_controls'] else 'No',
        'Yes' if spec['neutral_news'] else 'No',
        spec['n'],
        model.rsquared,
        spec['mean'],
    ])

results_display = pd.DataFrame(table_data, index=rows)
results_display


C:\Users\Hex\AppData\Roaming\Python\Python314\site-packages\statsmodels\regression\linear_model.py:1884: RuntimeWarning: invalid value encountered in sqrt
  return np.sqrt(np.diag(self.cov_params()))


,(1),(2),(3),(4),(5),(6)
Pro-Party News,0.092,0.088,0.041,0.037,NaN,0.077
Pro-Party News SE,0.006,NaN,69.103,0.007,NaN,NaN
Partisanship x Pro-Party,NaN,NaN,0.099,NaN,NaN,NaN
Partisanship x Pro-Party SE,NaN,NaN,62.373,NaN,NaN,NaN
Anti-Party News,NaN,NaN,NaN,-0.048,NaN,NaN
Anti-Party News SE,NaN,NaN,NaN,0.007,NaN,NaN
True News,NaN,NaN,NaN,NaN,-0.059,-0.034
True News SE,NaN,NaN,NaN,NaN,133.482,NaN
Question FE,Yes,Yes,Yes,No,Yes,Yes
Round FE,Yes,Yes,Yes,Yes,Yes,Yes
